# Sobrevivencia de negocios por estado

La misma pregunta que `sobrevivencia.ipynb` (zona metropolitana de Guadalajara), para un estado completo: ¿qué negocios de DENUE 2019_11 siguen en 2024_11, y les va mejor a los que tienen vecinos de su giro?

Plantilla: `por_estado.py` la ejecuta con `ESTADO=NN` y guarda la copia ejecutada en `estados/` y las cifras en `resultados/`. Sin contexto por AGEB (el dataset de vivienda solo existe para la zona metropolitana de Guadalajara). Ninguna probabilidad pasa a Mapo (RNF5).

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sobrevivencia import datos, graficas, modelos, rasgos

ESTADO = os.environ.get("ESTADO", "14")
RAIZ = Path("../..")
EDICIONES = RAIZ / "mapo_core/data/denue_ediciones"
CLASES = RAIZ / "mapo_core/src/mapo_core/rubros/clases.csv"
RESULTADOS = Path("resultados")
CORTES = ["2018_11", "2019_11", "2023_11", "2024_11"]
BASE, FIN = "2019_11", "2024_11"
ESTADO

'03'

## $\color{#dda}{\text{1. Ediciones y cortes censales}}$

Si las bajas de este estado también llegan con los Censos Económicos, 2018 a 2019 y 2023 a 2024 caen y 2019 a 2023 no.

In [2]:
rutas = {e: datos.bajar(e, EDICIONES, ESTADO) for e in CORTES}
consecutivas = datos.persistencia(rutas)
consecutivas

,de,a,negocios,siguen
0,2018_11,2019_11,35006,58.1
1,2019_11,2023_11,35925,89.6
2,2023_11,2024_11,35948,68.6


## $\color{#dda}{\text{2. Tabla base}}$

In [3]:
base, fin = datos.leer(rutas[BASE]), datos.leer(rutas[FIN])
catalogo = datos.rubros(CLASES, fin)
df = datos.tabla_base(base, fin, None, catalogo, datos.caja_robusta(base))
df["meses_alta"] = datos.meses_desde_alta(df["fecha_alta"], BASE)
print(f"Sin coordenadas o fuera del estado: {len(base) - len(df):,} de {len(base):,}")
print(f"Giros sin clase en la tabla de Mapo: {df['sector'].isna().mean():.2%}")
df[["sigue", "sigue_o_reregistrado"]].mean().mul(100).round(1).rename("%").to_frame().assign(negocios=len(df))

Sin coordenadas o fuera del estado: 1 de 35,925
Giros sin clase en la tabla de Mapo: 1.26%


,%,negocios
sigue,61.5,35924
sigue_o_reregistrado,62.8,35924


In [4]:
df = df.join(rasgos.vecindad(df))
df["bloque"] = rasgos.bloques(df)
df["log_vecinos_mismo"] = np.log1p(df["vecinos_mismo_giro"])
df["log_vecinos_todos"] = np.log1p(df["vecinos_todos"])
razon = rasgos.agrupacion_por_giro(df)
df["giro_se_agrupa"] = df["grupo_giro"].map(razon < 0.8).fillna(False).astype(int)
df["vecinos_mismo_x_se_agrupa"] = df["log_vecinos_mismo"] * df["giro_se_agrupa"]
print(f"Giros con al menos 30 negocios: {len(razon)}; se agrupan: {(razon < 0.8).sum()}")

Giros con al menos 30 negocios: 140; se agrupan: 79


## $\color{#dda}{\text{3. Tasas a simple vista}}$

In [5]:
t = modelos.tasas(df.assign(estrato=pd.Categorical(df["estrato"], datos.ESTRATOS, ordered=True)), "sigue", "estrato")
graficas.barras_tasa(t, "estrato", f"Estado {ESTADO}: siguen en 2024, por personal ocupado en 2019", "Personal ocupado").show()
t

,estrato,negocios,sigue
0,0 a 5 personas,28270,58.7
1,6 a 10 personas,3726,70.9
2,11 a 30 personas,2763,72.0
3,31 a 50 personas,489,74.2
4,51 a 100 personas,331,76.4
5,101 a 250 personas,215,74.9
6,251 y más personas,130,72.3


## $\color{#dda}{\text{4. ¿Les va mejor a los que se juntan?}}$

Probabilidad lineal con efecto fijo por giro y errores agrupados por bloques de 2 km, en puntos porcentuales.

In [6]:
NUMERICAS = ["log_vecinos_mismo", "log_vecinos_todos", "en_plaza", "semifijo", "meses_alta"]
efecto = modelos.efecto_dentro_del_giro(df, "sigue", NUMERICAS)
efecto

,puntos,desde,hasta,p
log_vecinos_mismo,0.262,-1.040,1.564,0.693
log_vecinos_todos,0.013,-0.793,0.819,0.975
en_plaza,-1.646,-6.354,3.062,0.493
semifijo,-3.705,-7.606,0.195,0.063
meses_alta,0.210,0.197,0.224,0.000
estrato_101 a 250 personas,10.123,2.328,17.917,0.011
estrato_11 a 30 personas,11.188,8.835,13.541,0.000
estrato_251 y más personas,16.108,8.958,23.258,0.000
estrato_31 a 50 personas,11.834,8.461,15.207,0.000
estrato_51 a 100 personas,11.506,6.411,16.602,0.000


In [7]:
rereg = modelos.efecto_dentro_del_giro(df, "sigue_o_reregistrado", NUMERICAS)
interaccion = modelos.efecto_dentro_del_giro(df, "sigue", NUMERICAS + ["vecinos_mismo_x_se_agrupa"])
pd.concat({"con re-registros": rereg.loc[["log_vecinos_mismo"]],
           "interacción": interaccion.loc[["log_vecinos_mismo", "vecinos_mismo_x_se_agrupa"]]})

puntos  desde  hasta      p
con re-registros log_vecinos_mismo           0.612 -0.622  1.846  0.331
interacción      log_vecinos_mismo          -1.091 -3.607  1.425  0.395
                 vecinos_mismo_x_se_agrupa   1.831 -0.486  4.148  0.121

## $\color{#dda}{\text{5. ¿Qué tanto se puede predecir?}}$

In [8]:
metricas, _ = modelos.validar(df, "sigue", NUMERICAS)
resumen_metricas = metricas.groupby("modelo")[["auc", "brier"]].mean().round(3)
resumen_metricas

,auc,brier
modelo,,
logística,0.678,0.215
solo el giro (en muestra),0.630,0.224
árboles,0.686,0.213


## $\color{#dda}{\text{6. Cifras para el resumen nacional}}$

In [9]:
def fila(tabla, clave):
    return {k: float(tabla.loc[clave, k]) for k in ("puntos", "desde", "hasta")}

cifras = {
    "estado": ESTADO,
    "negocios": int(len(df)),
    "sigue": round(float(df["sigue"].mean()) * 100, 2),
    "sigue_o_reregistrado": round(float(df["sigue_o_reregistrado"].mean()) * 100, 2),
    "persistencia": consecutivas.set_index("a")["siguen"].to_dict(),
    "vecinos_mismo": fila(efecto, "log_vecinos_mismo"),
    "vecinos_todos": fila(efecto, "log_vecinos_todos"),
    "en_plaza": fila(efecto, "en_plaza"),
    "vecinos_mismo_rereg": fila(rereg, "log_vecinos_mismo"),
    "interaccion": fila(interaccion, "vecinos_mismo_x_se_agrupa"),
    "giros_se_agrupan": int((razon < 0.8).sum()),
    "auc_arboles": float(resumen_metricas.loc["árboles", "auc"]),
    "auc_giro": float(resumen_metricas.loc["solo el giro (en muestra)", "auc"]),
}
RESULTADOS.mkdir(exist_ok=True)
(RESULTADOS / f"{ESTADO}.json").write_text(json.dumps(cifras, ensure_ascii=False, indent=1), encoding="utf-8")
cifras

{'estado': '03',
 'negocios': 35924,
 'sigue': 61.49,
 'sigue_o_reregistrado': 62.83,
 'persistencia': {'2019_11': 58.1, '2023_11': 89.6, '2024_11': 68.6},
 'vecinos_mismo': {'puntos': 0.262, 'desde': -1.04, 'hasta': 1.564},
 'vecinos_todos': {'puntos': 0.013, 'desde': -0.793, 'hasta': 0.819},
 'en_plaza': {'puntos': -1.646, 'desde': -6.354, 'hasta': 3.062},
 'vecinos_mismo_rereg': {'puntos': 0.612, 'desde': -0.622, 'hasta': 1.846},
 'interaccion': {'puntos': 1.831, 'desde': -0.486, 'hasta': 4.148},
 'giros_se_agrupan': 79,
 'auc_arboles': 0.686,
 'auc_giro': 0.63}